<a href="https://colab.research.google.com/github/joexner/roxene/blob/master/notebooks/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Authenticate to GCP on a custom runtime
!gcloud auth login --no-launch-browser

# Set your GCP Project ID
PROJECT_ID = 'roxene-0'
!gcloud config set project {PROJECT_ID}



You are running on a Google Compute Engine virtual machine.
It is recommended that you use service accounts for authentication.

You can run:

  $ gcloud config set account `ACCOUNT`

to switch accounts if necessary.

Your credentials may be visible to others with access to this
virtual machine. Are you sure you want to authenticate with
your personal account?

Do you want to continue (Y/n)?  Y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.

In [2]:
%cd
# Check if roxene directory exists, if not, clone it.
!if [ ! -d roxene ]; then git clone https://github.com/joexner/roxene.git; fi

/root
Cloning into 'roxene'...
remote: Enumerating objects: 2730, done.
remote: Counting objects: 100% (392/392), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 2730 (delta 269), reused 267 (delta 204), pack-reused 2338 (from 1)
Receiving objects: 100% (2730/2730), 1.38 MiB | 8.77 MiB/s, done.
Resolving deltas: 100% (1864/1864), done.


In [3]:
# Change directory to roxene and pull latest changes.
%cd ~/roxene
!git pull
!git log -1 --pretty="%ci: %s"

/root/roxene
Already up to date.
2026-09-01 23:00:15 -0400: Created using Colab


In [4]:
!pip install -e .

Obtaining file:///root/roxene
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 17.9 MB/s eta 0:00:00
  Building editable for roxene (pyproject.toml) ... done
  Created wheel for roxene: filename=roxene-0.1.0-py3-none-any.whl size=1103 sha256=68b6b9ab51da7a6e393932e3a13ec213fe00dbfa57f79a11128d1bfcf65d30a4
  Stored in directory: /tmp/pip-ephem-wheel-cache-te1y2daf/wheels/8f/15/57/e3c3bc579d018f54484d3a829962bca2159f7ee03498809a64
Successfully built roxene


In [8]:
import subprocess

CLUSTER_NAME = 'roxene-alloy-db-1'
REGION = 'us-central1'
INSTANCE_NAME = 'primary'
DB_PASSWORD = 'enexor'
DB_NAME = 'postgres'  # Using default database

print(f"Fetching connection info for '{INSTANCE_NAME}' via PSC (Internal VPC)...")

# Get the internal PSC DNS name
db_host = subprocess.check_output(
    f"gcloud alloydb instances describe {INSTANCE_NAME} --cluster={CLUSTER_NAME} --region={REGION} --format='value(pscInstanceConfig.pscDnsName)'",
    shell=True, text=True
).strip()

# Some libraries struggle with the trailing dot of a Fully Qualified Domain Name.
if db_host.endswith('.'):
    db_host = db_host[:-1]

DB_HOST = db_host
print(f"AlloyDB Internal Host (PSC DNS): {DB_HOST}")


Fetching connection info for 'primary' via PSC (Internal VPC)...
AlloyDB Internal Host (PSC DNS): 8812edaa-c0b9-4306-8c6a-6a31a1162e86.462c950b-e8dc-456a-92b6-602c9718c175.us-central1.alloydb-psc.goog


In [10]:
# Install the AlloyDB Python Connector and its dependencies
!pip install "google-cloud-alloydb-connector[pg8000]" SQLAlchemy -q

import time
import sqlalchemy
from google.cloud.alloydb.connector import Connector, IPTypes

# Dynamically generate a new database name
DB_NAME = f"roxene_db_{int(time.time())}"
print(f"Dynamically creating new database '{DB_NAME}' via AlloyDB Python Connector...")

INSTANCE_URI = f"projects/{PROJECT_ID}/locations/{REGION}/clusters/{CLUSTER_NAME}/instances/{INSTANCE_NAME}"

# Initialize Connector
connector = Connector()

def getconn():
    conn = connector.connect(
        INSTANCE_URI,
        "pg8000",
        user="postgres",
        password=DB_PASSWORD,
        db="postgres",
        ip_type=IPTypes.PSC
    )
    return conn

# Create SQLAlchemy engine with AUTOCOMMIT for creating a database
engine = sqlalchemy.create_engine(
    "postgresql+pg8000://",
    creator=getconn,
    isolation_level="AUTOCOMMIT"
)

try:
    with engine.connect() as db_conn:
        db_conn.execute(sqlalchemy.text(f"CREATE DATABASE {DB_NAME}"))
    print(f"Successfully created database: {DB_NAME}")
except Exception as e:
    print(f"Error creating database {DB_NAME}:\n{e}")
    raise
finally:
    # Cleanup connector
    connector.close()


Dynamically creating new database 'roxene_db_1788376698' in cluster 'roxene-alloy-db-1'...
Error creating database roxene_db_1788376698:
could not translate host name "8812edaa-c0b9-4306-8c6a-6a31a1162e86.462c950b-e8dc-456a-92b6-602c9718c175.us-central1.alloydb-psc.goog" to address: Name or service not known



OperationalError: could not translate host name "8812edaa-c0b9-4306-8c6a-6a31a1162e86.462c950b-e8dc-456a-92b6-602c9718c175.us-central1.alloydb-psc.goog" to address: Name or service not known


In [ ]:
# Construct the DB URL and run the script
db_url = f"postgresql+psycopg2://postgres:{DB_PASSWORD}@{DB_HOST}:5432/{DB_NAME}"
print(f"Connecting to: {db_url}")

import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(threadName)s]\t- %(name)s: %(message)s', force=True)
logging.getLogger("roxene.tic_tac_toe.environment").setLevel(logging.DEBUG)

import sys
import runpy
import site
import importlib

# Refresh site packages and invalidate import caches so the kernel sees the new package
site.main()
importlib.invalidate_caches()

import roxene

sys.argv = ["roxene", "1000", "1000", "--num_threads", "20", "--db_url", db_url]
runpy.run_module("roxene.tic_tac_toe")
